# CORTEX-LM: auditoría reproducible del checkpoint (solo lectura)

Esta primera etapa **no entrena ni sube el modelo**. Descarga una copia, forma archivos independientes de train y holdout a partir de artículos distintos, y compara la predicción del checkpoint con un baseline Witten–Bell de bytes.

Revisa el manifiesto JSON y los hashes antes de decidir el siguiente paso. No se usará el holdout de esta auditoría para seleccionar hiperparámetros ni promover modelos. El motor corre en CPU/OpenMP; una GPU T4 no lo acelera.

1. Guarda `HF_TOKEN` en Colab Secrets; no lo pegues en una celda.
2. Ejecuta las celdas 1 a 7. Esta fase no modifica el checkpoint remoto.
3. Descarga y comparte `evaluacion_checkpoint.json` (no contiene credenciales).

In [ ]:
# Celda 1: leer HF_TOKEN desde Colab Secrets; nunca se muestra ni se escribe en el notebook
import os
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Configura HF_TOKEN en Colab Secrets; no lo pegues en el cuaderno.')
os.environ['HF_TOKEN'] = hf_token
del hf_token
!pip install -q huggingface_hub datasets


In [ ]:
# Celda 2: clonar/actualizar la rama y usar una sola copia fuente del repositorio
import os, subprocess
from pathlib import Path
BRANCH = os.environ.get('CORTEX_BRANCH', 'arena/b0c825e4-nuby')
REPO = Path('/content/nuby_repo')
URL = 'https://github.com/alexsndersoto04-source/nuby-.git'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
%cd /content/nuby_repo


In [ ]:
# Celda 3: compilar el motor CPU/OpenMP
!make -C cortex-lm portable -j2
!ls -lh cortex-lm/bin/cerebro


In [ ]:
# Celda 4: descargar el checkpoint
!python cloud/colab_sync.py download
!ls -lh cortex-lm/data/cerebro_macro_1m.bin


In [ ]:
# Celda 5: preparar train y holdout nuevos, sin artículos compartidos
from datasets import load_dataset
from pathlib import Path
import json, hashlib
SEED_SPLIT = 20261117
DATASET_ID = 'wikimedia/wikipedia'
DATASET_CONFIG = '20231101.es'
ds = load_dataset(DATASET_ID, DATASET_CONFIG, split='train', streaming=True).shuffle(seed=SEED_SPLIT, buffer_size=5000)
docs = iter(ds)
vistos = set()
def recoger_bytes(objetivo):
    partes, titulos, total = [], [], 0
    while total < objetivo:
        item = next(docs)
        titulo = str(item.get('title', '')).strip()
        texto = str(item.get('text', '')).strip()
        if not titulo or titulo in vistos or len(texto) < 300: continue
        restantes = objetivo - total
        raw = (titulo + '\n' + texto + '\n\n').encode('utf-8')
        trozo = raw[:restantes].decode('utf-8', 'ignore').encode('utf-8')
        if not trozo: continue
        partes.append(trozo); titulos.append(titulo); vistos.add(titulo); total += len(trozo)
    return b''.join(partes), titulos

train_bytes, titulos_train = recoger_bytes(200_000)
holdout_bytes, titulos_holdout = recoger_bytes(16_384)
assert not (set(titulos_train) & set(titulos_holdout)), 'Hay títulos compartidos entre train y holdout.'
Path('/content/train_wikipedia.txt').write_bytes(train_bytes)
Path('/content/holdout_wikipedia.txt').write_bytes(holdout_bytes)
metadata = {
    'dataset_id': DATASET_ID, 'config': DATASET_CONFIG, 'split': 'train streaming + shuffle buffer',
    'seed': SEED_SPLIT, 'shuffle_buffer': 5000, 'train_bytes': len(train_bytes),
    'holdout_bytes': len(holdout_bytes), 'train_titles': titulos_train, 'holdout_titles': titulos_holdout,
    'titles_disjoint': not bool(set(titulos_train) & set(titulos_holdout)),
}
Path('/content/split_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"Train: {len(train_bytes)} bytes, {len(titulos_train)} artículos; holdout: {len(holdout_bytes)} bytes, {len(titulos_holdout)} artículos")
print('Títulos disjuntos:', metadata['titles_disjoint'])


In [ ]:
# Celda 6: evaluación comparativa reproducible; solo lectura, sin actualizar el modelo
import subprocess
from pathlib import Path
repo = Path('/content/nuby_repo')
modelo = repo / 'cortex-lm/data/cerebro_macro_1m.bin'
binario = repo / 'cortex-lm/bin/cerebro'
train = Path('/content/train_wikipedia.txt')
holdout = Path('/content/holdout_wikipedia.txt')
metadata = Path('/content/split_metadata.json')
reporte = Path('/content/evaluacion_checkpoint.json')
for p in (modelo, binario, train, holdout, metadata):
    if not p.is_file(): raise FileNotFoundError(p)
cmd = ['python3', str(repo / 'cortex-lm/tools/evaluar_reproducible.py'),
       '--modelo', str(modelo), '--binario', str(binario), '--train', str(train),
       '--holdout', str(holdout), '--metadata-json', str(metadata),
       '--dataset-label', 'wikimedia/wikipedia:20231101.es', '--salida', str(reporte)]
resultado = subprocess.run(cmd, cwd=repo / 'cortex-lm', text=True, capture_output=True)
print(resultado.stdout)
if resultado.returncode: raise RuntimeError(resultado.stderr[-4000:])
print('Manifiesto listo:', reporte, '| bytes:', reporte.stat().st_size)


In [ ]:
# Celda 7: descargar solo el manifiesto de evaluación (no sube ni altera el checkpoint)
from google.colab import files
files.download('/content/evaluacion_checkpoint.json')

---
## Qué sigue

Esta ejecución solo mide el checkpoint existente. No entrenes todavía: comparte el manifiesto descargado para revisar el hash del modelo, las identidades de train/holdout y la diferencia frente al baseline. Si la evaluación es reproducible y suficientemente amplia, el siguiente experimento se hará sobre una copia y con un test final sellado que no se reutiliza para elegir cambios.

No compartas el token ni otros secretos.